[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/04_ambiguity/04_ambiguous_problems.ipynb)

# 04 · 模糊问题与需求澄清（澄清清单 / 逐步收敛 / 优先级打分 / 假设决策规则）

目标：把「面对没有边界的问题该怎么办」从一套经验之谈，变成几个**可以运行、可以断言**的小工具。

本 notebook 你会亲手实现：
1. **环境自检** —— 纯标准库 + numpy，本课全程不需要 GPU、不需要联网
2. **六维度澄清清单 + 问题分类器** —— 给一段对话，自动标出每句话覆盖了哪个维度
3. **覆盖度检查器**（✏️ 练习）—— 给一批已问的问题，判断六个维度覆盖了几个、漏了哪个
4. **从「一句话」到「有边界的题面」的逐步收敛演练**
5. **收敛声明生成器**（✏️ 练习）—— 自动生成「我先按 A 假设展开，需要的话再切 B」这句话
6. **影响×成本×不确定性打分排序器**（✏️ 练习）—— RICE 打分法的简化实现
7. **「假设并前进」的决策规则**（✏️ 练习）—— 含「卡在澄清阶段」的强制停止逻辑

> 心智模型：**澄清的目的不是把问题问到毫无歧义，而是问到「你敢为这个假设负责」为止。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)
assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'argsort')
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 六维度澄清清单与关键词分类器

清单本身是一个字典：维度 -> {展示名, 问题模板, 面向 TSR 的具体问法, 用于识别这句话属于哪个维度的关键词}。

分类器只做最简单的事：**按固定顺序扫描六个维度，命中第一个包含的关键词就归类**——
这就是面试里「一句话属于哪个维度」的判断逻辑，写成代码后可以拿真实对话去检验。

In [ ]:
DIM_ORDER = ['goal', 'user', 'constraint', 'baseline', 'success', 'non_goal']

CHECKLIST = {
    'goal':       {'label': '目标 Goal',        'ask': '“效果”具体指哪个指标？提升多少算达标？'},
    'user':       {'label': '用户 User',         'ask': '这次改进服务于谁、在什么场景下使用？'},
    'constraint': {'label': '约束 Constraint',   'ask': '延迟/算力/周期有什么限制？能不能重训模型？'},
    'baseline':   {'label': '现状 Baseline',     'ask': '现在的水平是多少？已知的短板在哪？'},
    'success':    {'label': '成功标准 Success',   'ask': '怎么算这次改进成功？谁来验收？'},
    'non_goal':   {'label': '不做什么 Non-goal',  'ask': '这次明确排除在外的是什么？'},
}
assert list(CHECKLIST) == DIM_ORDER

KEYWORDS = {
    'goal':       ['指标', '目标是什么', '要提升什么'],
    'user':       ['使用场景', '谁在用', '什么车型'],
    'constraint': ['延迟预算', '算力', '不能超过'],
    'baseline':   ['现在', '基线', '现状'],
    'success':    ['算成功', '验收标准', '达到多少算过关'],
    'non_goal':   ['不做', '范围之外', '这次不管'],
}

def classify_question(q, keywords=KEYWORDS, order=DIM_ORDER):
    """按固定顺序扫描六维度关键词，命中第一个即返回该维度；都不命中返回 None。"""
    for dim in order:
        for kw in keywords[dim]:
            if kw in q:
                return dim
    return None

for dim in DIM_ORDER:
    print(f"{CHECKLIST[dim]['label']:<18} 面试话术示例：{CHECKLIST[dim]['ask']}")
print('\n✅ 六维度清单就位。')

In [ ]:
# —— 用一段真实对话检验分类器 ——
# 面试官原话："我们想提升 TSR 的效果，你会怎么做？"
TRANSCRIPT = [
    "您说的'提升TSR效果'，具体是想提升哪个指标：mAP、召回还是误检率？",   # goal
    "这套系统主要在什么使用场景下跑：高速、城市道路，还是两者都要？",     # user
    "延迟预算和算力有没有限制？比如推理必须在多少毫秒内完成？",          # constraint
    "现在的基线是什么水平？现在的mAP或者召回大概是多少？",               # baseline
    "怎么算成功？验收标准是提升多少个点，还是要达到某个绝对值？",        # success
    "有没有什么是这次明确不做的，比如新增类别或者上线新硬件？",          # non_goal
]
tags = [classify_question(q) for q in TRANSCRIPT]
assert tags == DIM_ORDER, tags   # 六句话依次精确覆盖六个维度

for q, dim in zip(TRANSCRIPT, tags):
    print(f"[{CHECKLIST[dim]['label']:<18}] {q}")
print('\n✅ 分类器正确识别了每一句话所属的维度。')

## ✏️ 练习 1：覆盖度检查器

实现 `coverage_summary(questions)`，返回 `(covered, missing, ratio)`：
- `covered`：一个 `set`，包含被至少一个问题命中的维度
- `missing`：一个**列表**，按 `DIM_ORDER` 顺序列出没被覆盖的维度
- `ratio`：`len(covered) / len(DIM_ORDER)`，覆盖率

对每个问题调用 `classify_question`；分类结果为 `None` 的问题不计入覆盖。

In [ ]:
def coverage_summary(questions, keywords=KEYWORDS, order=DIM_ORDER):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
c_full, m_full, r_full = coverage_summary(TRANSCRIPT)
assert c_full == set(DIM_ORDER), c_full
assert m_full == [], m_full
assert abs(r_full - 1.0) < 1e-12, r_full

c_half, m_half, r_half = coverage_summary(TRANSCRIPT[:3])
assert c_half == {'goal', 'user', 'constraint'}, c_half
assert m_half == ['baseline', 'success', 'non_goal'], m_half
assert abs(r_half - 0.5) < 1e-12, r_half

# 连续问了三句话，看起来很勤快，但都只是"目标"维度的不同措辞 —— 打勾数应该只有 1，而不是 3
REPEATED = [
    "'提升效果'具体是指哪个指标？",
    "这个指标是不是主要看召回率？",
    "为什么这个指标对业务这么重要？",
]
c_rep, m_rep, r_rep = coverage_summary(REPEATED)
assert c_rep == {'goal'}, c_rep                  # 三句话全部落在同一维度，只打了一格勾
assert abs(r_rep - 1/6) < 1e-12, r_rep
assert len(m_rep) == 5, m_rep

print(f'完整六问覆盖率: {r_full:.2f}  缺失: {m_full}')
print(f'只问三问覆盖率: {r_half:.2f}  缺失: {m_half}')
print(f'看似问了3句不同的话, 实际覆盖率: {r_rep:.2f}（只覆盖 {sorted(c_rep)}）')
print('\n✅ 练习 1 通过：覆盖率看的是"打勾格数"，不是"提问次数"。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def coverage_summary(questions, keywords=KEYWORDS, order=DIM_ORDER):
    tagged = [classify_question(q, keywords, order) for q in questions]
    covered = set(d for d in tagged if d is not None)
    missing = [d for d in order if d not in covered]
    ratio = len(covered) / len(order)
    return covered, missing, ratio

## 2 · 从「一句话」到「有边界的题面」：逐步收敛演练

把第 1 节问出来的六个答案，串成一份「收敛后的题面」。这里先用一个 worked 版本走一遍，
下一节的练习会把「生成确认话术」这部分单独抽出来实现。

In [ ]:
# 面试官对六个问题的回答（假设是这样的对话结果）
ANSWERS = {
    'goal': '召回率',
    'user': '城市道路场景',
    'constraint': '这次不重训整个检测器（面试官没提到重训预算，我们主动做这个假设）',
    'baseline': '目前 80m 内限速标志召回大概 70%',
    'success': '召回提升且不新增误检',
    'non_goal': '不新增标志类别、不换传感器',
}
assert set(ANSWERS) == set(DIM_ORDER)

CONVERGENCE_TRACE = [
    ('open',     '提升 TSR 效果'),
    ('ask_goal', f"指标不明确，先问：具体是哪个指标？—— 回答：{ANSWERS['goal']}"),
    ('scope',    f"补场景与约束：{ANSWERS['user']}；{ANSWERS['constraint']}"),
    ('bounded',  f"半收敛题面：在{ANSWERS['user']}下，把 80m 内限速标志的{ANSWERS['goal']}"
                 f"从{ANSWERS['baseline']}提上去，且{ANSWERS['success']}"),
]
for step, text in CONVERGENCE_TRACE:
    print(f'[{step:<9}] {text}')

assert CONVERGENCE_TRACE[0][0] == 'open' and CONVERGENCE_TRACE[-1][0] == 'bounded'
print('\n✅ 从一句话走到了一个可以动手分析的题面。')

## ✏️ 练习 2：收敛声明生成器

实现 `confirm_statement(assumption, alternative)`，返回一句包含固定结构的确认话术：
`"我先按 {assumption} 这个假设展开；如果不对，需要的话我们再切到 {alternative}。"`

再实现 `bound_problem(goal, user, constraint, success, non_goals)`，把六维度的答案
（`non_goals` 是列表）拼成一段结构化的「有边界题面」字符串，要求依次包含
目标、场景、约束、成功标准，以及用「不做：」开头列出 `non_goals`（用顿号 `、` 连接）。

In [ ]:
def confirm_statement(assumption, alternative):
    # TODO
    raise NotImplementedError

def bound_problem(goal, user, constraint, success, non_goals):
    # TODO：返回类似
    # "目标：{goal}；场景：{user}；约束：{constraint}；成功标准：{success}；不做：{non_goals 用、连接}"
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s = confirm_statement('限速标志·80m·不新增误检', '聚焦别的标志类别或场景')
assert '先按' in s and '需要的话' in s, s
assert '限速标志·80m·不新增误检' in s and '聚焦别的标志类别或场景' in s, s
print(s)

p = bound_problem(
    goal='把 80m 内限速标志的召回率提升',
    user='城市道路场景',
    constraint='不重训整个检测器',
    success='召回提升且不新增误检',
    non_goals=['新增标志类别', '更换传感器'],
)
assert '目标：把 80m 内限速标志的召回率提升' in p, p
assert '场景：城市道路场景' in p, p
assert '约束：不重训整个检测器' in p, p
assert '成功标准：召回提升且不新增误检' in p, p
assert '不做：新增标志类别、更换传感器' in p, p
print('\n' + p)
print('\n✅ 练习 2 通过：一句"我先按……展开"+ 一段结构化题面，就是收敛的完整产出。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 2 参考答案
def confirm_statement(assumption, alternative):
    return f'我先按 {assumption} 这个假设展开；如果不对，需要的话我们再切到 {alternative}。'

def bound_problem(goal, user, constraint, success, non_goals):
    non_goal_text = '、'.join(non_goals)
    return (f'目标：{goal}；场景：{user}；约束：{constraint}；'
            f'成功标准：{success}；不做：{non_goal_text}')

## 3 · 影响 × 成本 × 不确定性打分排序（RICE 的简化版）

$$\text{Score} = \dfrac{\text{Impact} \times \text{Confidence}}{\text{Cost}}, \quad \text{Confidence} = 1-\text{Uncertainty}$$

下面先给出一组候选方向（对应正文第 4 节的例子），worked 一遍手算结果，再在练习里实现打分与排序函数。

In [ ]:
CANDIDATES = [
    {'name': '加多帧时序投票降低闪烁',           'impact': 7, 'confidence': 0.8, 'cost': 3},
    {'name': '难例挖掘：易混淆标志对(限速60/80)', 'impact': 6, 'confidence': 0.7, 'cost': 4},
    {'name': '扩充远处小目标训练数据',            'impact': 9, 'confidence': 0.5, 'cost': 7},
    {'name': '重新标注一批噪声标签',              'impact': 4, 'confidence': 0.6, 'cost': 5},
    {'name': '换用更强检测器架构',                'impact': 6, 'confidence': 0.4, 'cost': 9},
]

# 手算校验（对应正文表格）
manual_scores = {
    '加多帧时序投票降低闪烁': 7 * 0.8 / 3,
    '难例挖掘：易混淆标志对(限速60/80)': 6 * 0.7 / 4,
    '扩充远处小目标训练数据': 9 * 0.5 / 7,
    '重新标注一批噪声标签': 4 * 0.6 / 5,
    '换用更强检测器架构': 6 * 0.4 / 9,
}
assert abs(manual_scores['加多帧时序投票降低闪烁'] - 1.8667) < 1e-3
assert abs(manual_scores['难例挖掘：易混淆标志对(限速60/80)'] - 1.05) < 1e-9
for name, s in sorted(manual_scores.items(), key=lambda kv: -kv[1]):
    print(f'{s:.3f}  {name}')
print('\n✅ 手算校验通过，练习 3 要把这个计算过程写成可复用的函数。')

## ✏️ 练习 3：打分与排序函数

实现 `rice_score(impact, confidence, cost)` 与 `rank_candidates(candidates)`：
- `rice_score` 就是上面的公式
- `rank_candidates` 返回按 `score` **降序**排列的 `candidates` 副本（每个 dict 里加一个 `'score'` 键），
  分数相同（本例不会出现，但要写对）时按 `name` 字典序升序，保证结果确定

In [ ]:
def rice_score(impact, confidence, cost):
    # TODO
    raise NotImplementedError

def rank_candidates(candidates):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(rice_score(7, 0.8, 3) - 7*0.8/3) < 1e-12
assert abs(rice_score(6, 0.4, 9) - 6*0.4/9) < 1e-12

ranked = rank_candidates(CANDIDATES)
names_in_order = [c['name'] for c in ranked]
assert names_in_order == [
    '加多帧时序投票降低闪烁',
    '难例挖掘：易混淆标志对(限速60/80)',
    '扩充远处小目标训练数据',
    '重新标注一批噪声标签',
    '换用更强检测器架构',
], names_in_order
assert all('score' in c for c in ranked)
assert ranked[0]['score'] > ranked[1]['score'] > ranked[-1]['score']
# 原始列表不应被就地修改（返回副本）
assert 'score' not in CANDIDATES[0]

for i, c in enumerate(ranked, 1):
    print(f"{i}. {c['name']:<28} score={c['score']:.3f}")
print('\n✅ 练习 3 通过：排序结果与正文表格一致——时序投票第一，换架构垫底但不是被排除。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 3 参考答案
def rice_score(impact, confidence, cost):
    return impact * confidence / cost

def rank_candidates(candidates):
    scored = [dict(c, score=rice_score(c['impact'], c['confidence'], c['cost'])) for c in candidates]
    return sorted(scored, key=lambda c: (-c['score'], c['name']))

## 4 ·「假设并前进」的决策规则

核心判据：当 $\mathbb{E}[\text{Cost}_{\text{wrong}}] = P(\text{猜错}) \times \text{Cost}_{\text{if wrong}}$
明显大于「问一句」的代价时才问；否则假设并前进。**但一旦澄清用时超过预算的某个比例，
无论期望代价怎么算，都要强制停止提问、假设并前进**——这是"卡在澄清阶段不动也是扣分项"的代码化。

In [ ]:
def expected_cost_of_wrong_assumption(prob_wrong, cost_if_wrong):
    return prob_wrong * cost_if_wrong

# worked 演示：两种典型场景
e_high = expected_cost_of_wrong_assumption(prob_wrong=0.5, cost_if_wrong=10)   # 目标指标猜错
e_low  = expected_cost_of_wrong_assumption(prob_wrong=0.05, cost_if_wrong=10)  # 示例图要不要脱敏这种细节
assert e_high == 5.0 and e_low == 0.5
print(f'"目标指标猜错"的期望代价 = {e_high} —— 明显大于问一句的代价(约1) -> 应该问')
print(f'"示例要不要脱敏"的期望代价 = {e_low} —— 小于问一句的代价 -> 应该直接假设')

## ✏️ 练习 4：ask_or_assume 决策规则

实现 `ask_or_assume(prob_wrong, cost_if_wrong, cost_of_asking, time_used_clarify, total_budget, stall_frac=0.2)`：

1. **先检查是否已经卡关**：若 `time_used_clarify > stall_frac * total_budget`，
   无论期望代价多大，直接返回 `'assume_and_proceed'`（这是"卡在澄清阶段"的强制熔断）。
2. 否则计算 `E = prob_wrong * cost_if_wrong`：
   - `E > cost_of_asking` → 返回 `'ask'`
   - 否则 → 返回 `'assume_and_proceed'`

In [ ]:
def ask_or_assume(prob_wrong, cost_if_wrong, cost_of_asking,
                   time_used_clarify, total_budget, stall_frac=0.2):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 1) 代价高、时间还早 -> 该问
assert ask_or_assume(prob_wrong=0.5, cost_if_wrong=10, cost_of_asking=1,
                      time_used_clarify=2, total_budget=45) == 'ask'

# 2) 代价低、时间还早 -> 假设并前进（不必为细枝末节耗时）
assert ask_or_assume(prob_wrong=0.05, cost_if_wrong=10, cost_of_asking=1,
                      time_used_clarify=2, total_budget=45) == 'assume_and_proceed'

# 3) 代价很高，但澄清已经用掉 12 分钟（> 45*0.2=9）-> 强制熔断，即使不确定也要往前推
assert ask_or_assume(prob_wrong=0.9, cost_if_wrong=10, cost_of_asking=1,
                      time_used_clarify=12, total_budget=45) == 'assume_and_proceed'

# 4) 恰好卡在阈值边界：9 分钟（== 45*0.2）不算超过，仍按期望代价判断
assert ask_or_assume(prob_wrong=0.5, cost_if_wrong=10, cost_of_asking=1,
                      time_used_clarify=9, total_budget=45) == 'ask'

# 5) 更短的总预算（比如 25 分钟一题）下，熔断点相应提前
assert ask_or_assume(prob_wrong=0.9, cost_if_wrong=10, cost_of_asking=1,
                      time_used_clarify=6, total_budget=25, stall_frac=0.2) == 'assume_and_proceed'

for label, kwargs in [
    ('目标指标不明确·时间还早',   dict(prob_wrong=0.5, cost_if_wrong=10, cost_of_asking=1, time_used_clarify=2, total_budget=45)),
    ('示例细节·时间还早',         dict(prob_wrong=0.05, cost_if_wrong=10, cost_of_asking=1, time_used_clarify=2, total_budget=45)),
    ('高度不确定但已问了12分钟',  dict(prob_wrong=0.9, cost_if_wrong=10, cost_of_asking=1, time_used_clarify=12, total_budget=45)),
]:
    print(f'{label:<22} -> {ask_or_assume(**kwargs)}')
print('\n✅ 练习 4 通过：即使还不确定，卡关超过预算的 20% 就必须假设并前进。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 4 参考答案
def ask_or_assume(prob_wrong, cost_if_wrong, cost_of_asking,
                   time_used_clarify, total_budget, stall_frac=0.2):
    if time_used_clarify > stall_frac * total_budget:
        return 'assume_and_proceed'
    expected_cost_wrong = prob_wrong * cost_if_wrong
    if expected_cost_wrong > cost_of_asking:
        return 'ask'
    return 'assume_and_proceed'

---
## 🧪 真实工程胶囊：模糊需求收口的口播模板（中英对照）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 拿到模糊问题后的动作顺序（背这个顺序，不用背具体话术）
# ══════════════════════════════════════════════════════════════════════
# 1. 复述一遍，明说"这句话里有几处不清楚"
# 2. 按六维度清单问 4-6 个问题：目标/用户/约束/现状/成功标准/不做什么
#    —— 打勾看维度覆盖数，不看提问条数
# 3. 面试官说"你定"时，主动收口，声明一个假设
# 4. 列出候选方向，用 影响×置信度÷成本 排序，念出排序理由
# 5. 给分阶段计划，主动收尾并留一句"如果假设不对，我们随时调整"

# ══════════════════════════════════════════════════════════════════════
# B. 收敛确认话术（中 / EN —— JD 是英文岗位，两套都要能脱口而出）
# ══════════════════════════════════════════════════════════════════════
# CN: "我先按 A 这个假设往下展开；如果不对，需要的话我们再切到 B。"
# EN: "I'll start with assumption A and go from there;
#      if that's off, we can switch to B."
#
# CN: "您提到的这几点里，'效果'这个词最模糊，我想先确认一下具体指哪个指标。"
# EN: "Of everything you mentioned, 'improve the result' is the vaguest part —
#      let me first confirm which metric that refers to."
#
# CN: "为了不占用太多时间，我先做一个假设：这次不涉及重训整个检测器。
#      如果预算其实更宽松，请随时打断我。"
# EN: "To keep us moving, I'll assume this doesn't involve retraining
#      the whole detector. Please interrupt if that budget is actually there."

# ══════════════════════════════════════════════════════════════════════
# C. 何时该问 / 何时该假设（判据速查）
# ══════════════════════════════════════════════════════════════════════
# 猜错代价高 (方向性错误/安全相关)         -> 问
# 猜错代价低 (实现细节)                   -> 直接假设，说出来即可
# 已经问了 3 轮，对方仍说"你定"           -> 停止追问，主动拍板
# 澄清耗时已超过总预算的 ~20%             -> 无论多不确定都必须停止提问

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 通用四步框架 (澄清/分解/假设验证/收敛权衡)     -> C65-00
# · 估算题与数量级心算                            -> C65-01
# · 诊断与归因推理（技术问题的排查，不是需求澄清）  -> C65-02
# · 权衡与决策（可逆/不可逆决策速度、多维打分）     -> C65-03（本模块的"假设并前进"与其同源）
# · ML 系统设计里的正式需求澄清格式               -> C63-01
# · 把本模块能力放进完整白板模拟演练               -> C65-05（下一模块）
'''
print(RECIPE)
for token in ['先按', 'assumption A', '你定', '20%', 'C65-00', 'C63-01']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：动作顺序 / 中英确认话术 / 问-假设判据 / 课程分工')

### 小结

- **面对没有边界的问题，面试官考的不是技术方案，是你能不能先把问题变得有边界。**
  过早收敛（不问就假设）和过度澄清（问到面试结束）是同一个错误的两个极端。
- **六维度清单**（目标/用户/约束/现状/成功标准/不做什么）用来检查澄清的**覆盖面**，
  打勾数看的是覆盖了几个正交维度，不是问了几个问题——反复问同一维度的变体不算"问得系统"。
- **收敛的标准动作是「假设 + 声明 + 请确认」**：「我先按 A 这个假设展开，需要的话再切 B」
  这句话同时做了显式声明假设、保留退出通道、不停下来等答案三件事。
- **多个候选方向时，用 影响 × 置信度 ÷ 成本 排序**，把「我觉得应该先做 A」变成可以被挑战、
  可以被复算的结论；数字不必精确，但排序逻辑必须能讲清楚。
- **卡在澄清阶段不动本身就是扣分项。** 猜错代价高、时间还早时才问；一旦澄清耗时超过预算的
  某个比例（notebook 里用 20% 作为经验阈值），无论多不确定，都要假设并前进——
  这比"假设猜错但主动修正"要糟糕得多，因为它说明你无法在信息不全时做出任何决定。

下一站：**模块 05 · 白板沟通与模拟演练**——把本模块学到的收敛能力，
放进边想边说、结构化表达、10 道开放题的完整模拟里。